Task 3: Trying ways to learn concept Z: 'Populism'   
   
Import PyToch and OpenAI CLIP to analyze text and images (We can use these libraries to analyze the images we generated in Task 2)

In [1]:
import torch
import clip
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
import csv

Load CLIP model and images from 'Populism Images'

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

image_folder = "populism images"
images = []
filenames = []

for root, dirs, files in os.walk(image_folder):
    for fname in files:
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            path = os.path.join(root, fname)
            try:
                img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
                images.append(img)
                filenames.append(os.path.relpath(path, image_folder))
            except Exception as e:
                print(f"Skipping {path}: {e}")

print(f"Loaded {len(images)} images")

Skipping populism images/Seed 3 Tree/seed32c.png: cannot identify image file 'populism images/Seed 3 Tree/seed32c.png'
Skipping populism images/Seed 2 Tree/seed231b.png: cannot identify image file 'populism images/Seed 2 Tree/seed231b.png'
Loaded 28 images


Extract image embeddings

In [3]:
with torch.no_grad():
    embeddings = [model.encode_image(img).cpu().numpy() for img in images]

embeddings = np.vstack(embeddings)
print("Embedding matrix shape:", embeddings.shape)

Embedding matrix shape: (28, 512)


Extract Prompts and Tones


In [6]:
prompt_to_level = {}

for root, dirs, files in os.walk(image_folder):
    for fname in files:
        if fname.lower().endswith(".csv"):
            path = os.path.join(root, fname)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        prompt = row["prompt"].strip()
                        level = int(row["populism_level_numeric"])
                        prompt_to_level[prompt] = level
            except Exception as e:
                print(f"Skipping {path}: {e}")

print(f"Loaded {len(prompt_to_level)} prompt to level mappings")
for prompt, level in prompt_to_level.items():
    print(f"  [{level}/5] {prompt[:80]}...")

Loaded 23 prompt to level mappings
  [1/5] Generate a social media political graphic using Seed 1 style. Tone: low populist...
  [2/5] Generate a newspaper-style illustration of community members discussing economic...
  [2/5] Generate an image that captures the political concept of populist at a 2/5 level...
  [1/5] can you come up with a social media post (image) that would contain a 1/5 level ...
  [2/5] Generate a social media political graphic using Seed 1 style. Tone: moderate pop...
  [4/5] Generate a social media political graphic using Seed 1 style. Tone: heavy populi...
  [4/5] Generate a social media political graphic using Seed 1 style. Tone: heavy populi...
  [2/5] Generate a social media political graphic using Seed 1 style. Tone: moderate pop...
  [1/5] Generate a social media political graphic using Seed 1 style. Tone: light populi...
  [3/5] Generate an image using Seed 2 as a reference, but make the populist themes more...
  [5/5] Using Seed 2 as a reference, modify t

Allow CLIP to assign a populism level to each processed image using a defined template

In [7]:
# Taken from model descriptions of an example of each of the 6 levels of populism in the dataset
level_descriptions = {
    0: "a formal technocratic policy briefing with experts and professionals, no populist imagery",
    1: "a quiet community listening session, politician taking notes, calm collaborative discussion",
    2: "a local town hall meeting with everyday citizens, mild people-centered tone",
    3: "a small rally with working families, people-first messaging, community-focused energy",
    4: "a large outdoor rally, raised fists, strong people-first rhetoric, energized crowd",
    5: "a massive protest outside a government building, people vs powerful framing, confrontational crowd",
}

level_texts = [level_descriptions[i] for i in range(len(level_descriptions))]
text_tokens = clip.tokenize(level_texts, truncate=True).to(device)

with torch.no_grad():
    level_features = model.encode_text(text_tokens).cpu().numpy()

image_levels = []
image_level_prob = []

for i, fname in enumerate(filenames):
    sims = cosine_similarity(embeddings[i:i+1], level_features)[0]
    prob = np.exp(sims) / np.exp(sims).sum()
    assigned_level = np.argmax(sims)
    image_levels.append(assigned_level)
    image_level_prob.append(prob)